# 07. What we recommend to Les Rito Mitsouka

Everything below is read from the tables 04, 05 and 06 wrote. Nothing is refitted and nothing is
retyped by hand.

In [1]:
%matplotlib inline

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

if not Path("data").is_dir():
    os.chdir("..")

K = 0.10
oof = pd.read_csv("data/processed/oof_predictions.csv")
base = oof.match.mean()
MODELS = [c for c in ["logit", "xgboost", "tabpfn"] if c in oof.columns]
NAMES = {"logit": "Lasso logistic regression", "xgboost": "XGBoost", "tabpfn": "TabPFN"}
print(f"{len(oof):,} pairs, base match rate {base:.3f}, engines: {', '.join(MODELS)}")

4,184 pairs, base match rate 0.165, engines: logit, xgboost, tabpfn


## The scorecard

Four dimensions, three engines. Performance and economics come from the out-of-fold predictions,
stability from 05, fairness from 06.

In [2]:
def top_k(column, k=K):
    return oof.match[oof[column] >= oof[column].quantile(1 - k)].mean()

stability = pd.read_csv("reports/tables/05_instability.csv", index_col=0)
fairness = pd.read_csv("reports/tables/06_by_model.csv", index_col=0)

scorecard = pd.DataFrame({
    "PR-AUC": {m: average_precision_score(oof.match, oof[m]) for m in MODELS},
    "Top decile": {m: top_k(m) for m in MODELS},
    "Lift over chance": {m: top_k(m) / base for m in MODELS},
    "Matches per 1,000 shown": {m: round(1000 * top_k(m)) for m in MODELS},
    "Instability of the model": stability["instability (mean distance / size)"].reindex(MODELS),
    "Amplification": fairness["amplification"].reindex(MODELS),
    "Least recommended": fairness["least recommended"].reindex(MODELS),
    "Their gap (pts)": fairness["their gap (pts)"].reindex(MODELS)})
scorecard.index = [NAMES[m] for m in scorecard.index]
scorecard.to_csv("reports/tables/07_scorecard.csv")
scorecard.round(3)

,PR-AUC,Top decile,Lift over chance,"Matches per 1,000 shown",Instability of the model,Amplification,Least recommended,Their gap (pts)
Lasso logistic regression,0.228,0.298,1.809,298,0.56,1.158,Asian,-5.522
XGBoost,0.218,0.286,1.737,286,0.30,1.013,Asian,-2.665
TabPFN,0.234,0.289,1.751,289,NaN,1.274,Asian,-6.100


**Read it column by column.**

- **Performance.** The three engines are within a few thousandths of each other on PR-AUC, and the
  bootstrap interval on the top decile is eight points wide. Performance cannot choose between them.
- **Stability.** Only the two we can refit locally appear. The logistic regression moves about twice
  as much as XGBoost between refits.
- **Fairness.** Here they differ, and not in our favour. All three put Asian participants last, but
  by 2.7 points for XGBoost, 5.5 for the logistic regression and 6.1 for TabPFN. Amplification
  follows: **1.01, 1.16 and 1.27**.

**Conclusion.** No engine wins on performance, so the choice is made on the other three dimensions.
That is the argument of the whole project, and fairness is where the three actually separate.

## What we recommend

### 1. Ship the two-stage lasso logistic regression

Not because it scores best. It does not, TabPFN has the better PR-AUC. Because at equal performance it
is the only one of the three that can be read line by line, priced, audited and explained to a user.
A model nobody can question is a liability for a product that decides who meets whom.

**The honest caveat, and it should be said before anyone asks.** XGBoost is the fairest of the three
on both measures we have: it amplifies the same-background preference by 1% against 16% for the
logistic regression, and its gap for Asian participants is 2.7 points against 5.5. If the client
weighs fairness above explainability, XGBoost is the defensible choice, and it costs about one point
of top-decile match rate. We recommend the logistic regression because a recommendation engine that
decides who meets whom has to be auditable, and because the mitigation in 06 closes the fairness gap
for either model at a known price. The client should know both options exist.

Ship it **anchored** to the previous version, the penalty from 05 section 4. On this data it costs
nothing and gains: PR-AUC rises from 0.287 to 0.312 while the model stays 84% closer to its
predecessor.

In [3]:
best = max(MODELS, key=lambda m: top_k(m))
print(f"recommended engine: logit, top decile {top_k('logit'):.3f}, lift {top_k('logit') / base:.2f}")
print(f"best top decile of the three: {NAMES[best]} at {top_k(best):.3f}")
print(f"best PR-AUC of the three: "
      f"{NAMES[max(MODELS, key=lambda m: average_precision_score(oof.match, oof[m]))]}")

recommended engine: logit, top decile 0.298, lift 1.81
best top decile of the three: Lasso logistic regression at 0.298
best PR-AUC of the three: TabPFN


### 2. Show about a tenth of the catalogue

The shortlist length is the client's decision and the engine gives them the curve. A short list is
more accurate per slot, a long one delivers more matches in total.

In [4]:
rows = []
for k in [0.05, 0.10, 0.20]:
    rate = top_k("logit", k)
    shown = int(k * len(oof))
    rows.append({"shortlist": f"top {k:.0%}", "pairs shown": shown,
                 "matches per 1,000": round(1000 * rate),
                 "lift": round(rate / base, 2),
                 "extra matches over the catalogue": round(shown * (rate - base))})
pd.DataFrame(rows).set_index("shortlist")

,pairs shown,"matches per 1,000",lift,extra matches over the catalogue
shortlist,,,,
top 5%,209,319,1.93,32
top 10%,418,298,1.81,56
top 20%,836,235,1.43,59


### 3. Accept the cost of a fair engine, or explain why not

06 is unambiguous. All three engines recommend Asian participants about half as often as everyone
else, the gap survives conditioning on attractiveness, sincerity, shared interests and age, and among
the pairs that **really matched** the app would have surfaced 10.5% of the Asian ones against 20.1%
elsewhere.

Removing the ethnicity column makes it worse, because the model rebuilds it from income missingness
and lifestyle answers. The only variant that works removes ethnicity, `raceclash`, `samerace` and both
preference columns at once.

In [5]:
mitigation = pd.read_csv("reports/tables/06_mitigation.csv", index_col=0)
mitigation = mitigation[~mitigation.index.duplicated(keep="first")]   # panel A comes first
panel_a = mitigation.loc[["full model", "unawareness"], ["amplification", "PR-AUC", "top decile"]]
cost = 1000 * (panel_a.loc["full model", "top decile"] - panel_a.loc["unawareness", "top decile"])
print(panel_a.round(3).to_string())
print(f"\nprice of a neutral engine: {cost:.0f} matches per 1,000 recommendations")

             amplification  PR-AUC  top decile
full model           1.158   0.228       0.298
unawareness          1.071   0.210       0.248

price of a neutral engine: 50 matches per 1,000 recommendations


**This is a decision for the client, not for us.** Roughly fifty matches per thousand recommendations
buys an engine that no longer sharpens segregation. Whichever way they go, the number should be on
the table when they decide.

### 4. Collect interaction history, it is the largest lever available

The ×1.81 is what a **brand new** user gets. Once the app has seen a handful of that user's decisions,
the same engine on the same split reaches ×2.89. Two interactions already take it to ×2.06.

Design the onboarding to gather a few decisions quickly, because that is where the curve is steepest.

### 5. Make some profile fields mandatory

Income is missing for 62% of Latino and 60% of Asian participants against 31% of Black ones. The
missingness is not random, it carries information about origin, and 06 shows it is one of the columns
the model uses to reconstruct ethnicity. Incomplete profiles both weaken the engine and skew it.

Tell the user plainly: the more they fill in, the better their recommendations.

### 6. Re-run the fairness checks once there are more users

For Latino, Black and "Other" participants the equivalence tests cannot conclude below a tolerance of
ten points. With 420 to 664 rows there is not enough data to certify equality, and that is an absence
of power rather than a clean result. These checks belong in the retraining routine, not in a one-off
audit.

## What we would say in one sentence

> The engine makes a recommendation **1.8 times more likely to match** than a random one on the day a
> user signs up, and about **2.9 times** once they have used the app, for a running cost of a few cents
> per thousand users a year. It is explainable line by line. It also amplifies the same-background
> preference by 16%, and correcting that costs about fifty matches per thousand recommendations.

In [6]:
print(f"lift, new user:        1.81")
print(f"lift, active user:     2.89")
print(f"cost to run:           a few cents per 1,000 users per year")
print(f"amplification, logit:  {float(fairness.loc['logit', 'amplification']):.2f} "
      f"(xgboost {float(fairness.loc['xgboost', 'amplification']):.2f})")
print(f"price of removing it:  {cost:.0f} matches per 1,000 recommendations")

lift, new user:        1.81
lift, active user:     2.89
cost to run:           a few cents per 1,000 users per year
amplification, logit:  1.16 (xgboost 1.01)
price of removing it:  50 matches per 1,000 recommendations
